In [210]:
import pandas as pd
from operator import itemgetter
import networkx as nx
import numpy as np

In [211]:
data = pd.read_csv('pre_survey.csv')

### Nodes

In [212]:
nodes = data[['ID','Name']] #only need these columns, we fit everything to name for ease (only 40 people)
nodes = nodes.rename(columns={'Name':'Label'})
print("There are", len(nodes), "nodes.")

There are 39 nodes.


In [213]:
nodes.to_csv('nodes.csv', index=False)

### Edges

In [214]:
working = data.iloc[:,4:] #removes this column
working = working.set_index('Name').drop(columns=['NetID','Last modified time']) #set index for stacking, remove other cols
working = working.stack().rename_axis(['Source','Target']).reset_index() #makes into a stack from matrix
edges = working.rename(columns={0:'Weight'}) #rename columns
edges = edges[edges['Target'].isin(nodes['Label'])] #only include targets in the club
edges.head()

,Source,Target,Weight
0,Nikhil Chinchalkar,Nikhil Chinchalkar,I am this person
1,Nikhil Chinchalkar,Jason Wang,I speak with them at least once a week
2,Nikhil Chinchalkar,Rithya Sriram,I speak with them at least once a week
3,Nikhil Chinchalkar,Carina Lau,I speak with them at least once a week
4,Nikhil Chinchalkar,Jenny Williams,I speak with them at least once a week


In [215]:
len(edges['Source'].unique()), len(edges['Target'].unique()) #sanity check, should be 40 people still

(39, 39)

In [216]:
edges['Weight'].unique() #used for mapping weights below

array(['I am this person', 'I speak with them at least once a week',
       "I've spoken to them more than once before",
       'I recognize their face/name', "I've spoken to them once before",
       "I've never seen/heard of this person before",
       'I speak with them everyday'], dtype=object)

In [217]:
weights_map = {'I am this person':0,
               'I speak with them everyday':4,
               'I speak with them at least once a week':3,
               "I've spoken to them more than once before":2,
               "I've spoken to them once before":1,
               'I recognize their face/name':0,
               "I've never seen/heard of this person before":0} #subject to change

In [218]:
edges['Weight'] = edges['Weight'].map(lambda x: weights_map[x])

In [219]:
edges.to_csv('edges.csv', index=False)

### Demographics

In [220]:
demographics = pd.read_csv('demographics.csv')
demographics = pd.merge(demographics, nodes, left_on='Full Name', right_on='Label', how='right')
demographics.to_csv('node_demographics.csv', index=False)

### Graph

In [221]:
undirected_edges = pd.merge(edges, edges, left_on=['Target','Source'], right_on=['Source','Target'], how='left')

In [222]:
undirected_edges['Average Weight'] = np.nanmean(undirected_edges[['Weight_x','Weight_y']], axis=1)
undirected_edges = undirected_edges.rename(columns={'Source_x':'Source'})
undirected_edges = undirected_edges.rename(columns={'Target_x':'Target'})
undirected_edges = undirected_edges[['Source', 'Target','Average Weight']]

In [223]:
undirected_edges = pd.concat([pd.DataFrame(np.sort(undirected_edges[['Source','Target']], axis=1), columns=['Source','Target']), 
           undirected_edges['Average Weight']], axis=1).drop_duplicates()

In [224]:
undirected_edges = undirected_edges[undirected_edges['Average Weight'] != 0]
undirected_edges = undirected_edges.rename(columns={'(Source,)':'Source'})
undirected_edges = undirected_edges.rename(columns={'(Target,)':'Target'})

In [225]:
undirected_edges

,Source,Target,Average Weight
1,Jason Wang,Nikhil Chinchalkar,3.0
2,Nikhil Chinchalkar,Rithya Sriram,3.0
3,Carina Lau,Nikhil Chinchalkar,3.0
4,Jenny Williams,Nikhil Chinchalkar,3.0
5,Nikhil Chinchalkar,Rahi Dasgupta,3.0
...,...,...,...
1322,Adam Azevedo,Rahi Dasgupta,1.0
1329,Carina Lau,Isabella Reyes-Famous,0.5
1338,Emi Labbe,Isabella Reyes-Famous,0.5
1368,Carina Lau,Emi Labbe,0.5


In [226]:
G_undirected = nx.from_pandas_edgelist(
    undirected_edges,
    source='Source',
    target='Target',
    edge_attr=['Average Weight'])

In [227]:
for _, row in demographics.iterrows():
    node_id = row['Full Name']

    G_undirected.add_node(node_id)
    for col in demographics.columns:
        if col != 'Full Name':
            G_undirected.nodes[node_id][col] = row[col]

In [228]:
nx.write_gexf(G_undirected, "undirected_cdj_network.gexf")

In [229]:
edges = edges[edges['Weight'] != 0]

In [230]:
G = nx.from_pandas_edgelist(
    edges,
    source='Source',
    target='Target',
    edge_attr=['Weight'],
    create_using=nx.DiGraph())

In [231]:
for _, row in demographics.iterrows():
    node_id = row['Full Name']

    G.add_node(node_id)
    for col in demographics.columns:
        if col != 'Full Name':
            G.nodes[node_id][col] = row[col]

In [232]:
nx.write_gexf(G, "cdj_network.gexf")

### Metrics

In [233]:
from matplotlib import pyplot as plt

In [234]:
density = nx.density(G_undirected)
print("Network density:", density)

Network density: 0.2807017543859649


In [235]:
print(nx.is_connected(G_undirected))

True


In [236]:
diameter = nx.diameter(G_undirected)
print("Network diameter:", diameter)

Network diameter: 3


In [258]:
nx.average_clustering(G_undirected, count_zeros=False)

0.6727236120996997

In [237]:
pair_dict = dict(nx.all_pairs_all_shortest_paths(G_undirected, weight=None))
all_paths = []
for person in pair_dict.keys():
    for target in pair_dict[person]:
        for pair in pair_dict[person][target]:
            all_paths.append(pair)

In [238]:
triadic_closure = nx.transitivity(G_undirected)
print("Triadic closure:", triadic_closure)

Triadic closure: 0.5223171889838556


In [239]:
degree_dict = dict(G_undirected.degree(G_undirected.nodes()))
sorted_degree = sorted(degree_dict.items(), key=itemgetter(1), reverse=True)

In [240]:
betweenness_dict = nx.betweenness_centrality(G) # Run betweenness centrality
eigenvector_dict = nx.eigenvector_centrality(G) # Run eigenvector centrality
sorted_betweenness = sorted(betweenness_dict.items(), key=itemgetter(1), reverse=True)
sorted_eigenvector = sorted(eigenvector_dict.items(), key=itemgetter(1), reverse=True)
# sorted_eigenvector